<a href="https://colab.research.google.com/github/asipnana/ProjectNLP/blob/main/notebooks/02_preprocessing_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [88]:
%cd /content/ProjectNLP

/content/ProjectNLP


In [89]:
!git pull

remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 39 (delta 14), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (39/39), 747.18 KiB | 3.09 MiB/s, done.
From https://github.com/asipnana/ProjectNLP
   9731710..bf1dd3c  main       -> origin/main
Updating 9731710..bf1dd3c
Fast-forward
 README.md                                          |   4 +-
 .../raw/PRDECT-ID Dataset.csv                      |   0
 notebooks/01_eda_preprocessing.ipynb               | 870 ++++++++++++++++++++-
 3 files changed, 832 insertions(+), 42 deletions(-)
 rename PRDECT-ID Dataset.csv => dataset/raw/PRDECT-ID Dataset.csv (100%)


In [90]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [92]:
df = pd.read_csv("/content/ProjectNLP/dataset/raw/PRDECT-ID Dataset.csv")

In [93]:
df.head()

,Category,Product Name,Location,Price,Overall Rating,Number Sold,Total Review,Customer Rating,Customer Review,Sentiment,Emotion
0,Computers and Laptops,Wireless Keyboard i8 Mini TouchPad Mouse 2.4G ...,Jakarta Utara,53500,4.9,5449,2369,5,Alhamdulillah berfungsi dengan baik. Packaging...,Positive,Happy
1,Computers and Laptops,PAKET LISENSI WINDOWS 10 PRO DAN OFFICE 2019 O...,Kota Tangerang Selatan,72000,4.9,2359,1044,5,"barang bagus dan respon cepat, harga bersaing ...",Positive,Happy
2,Computers and Laptops,SSD Midasforce 128 Gb - Tanpa Caddy,Jakarta Barat,213000,5.0,12300,3573,5,"barang bagus, berfungsi dengan baik, seler ram...",Positive,Happy
3,Computers and Laptops,ADAPTOR CHARGER MONITOR LCD LED TV LG merek LG...,Jakarta Timur,55000,4.7,2030,672,5,bagus sesuai harapan penjual nya juga ramah. t...,Positive,Happy
4,Computers and Laptops,ADAPTOR CHARGER MONITOR LCD LED TV LG merek LG...,Jakarta Timur,55000,4.7,2030,672,5,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",Positive,Happy


In [94]:
df = df[['Customer Review', 'Sentiment', 'Emotion']]
df.head()

,Customer Review,Sentiment,Emotion
0,Alhamdulillah berfungsi dengan baik. Packaging...,Positive,Happy
1,"barang bagus dan respon cepat, harga bersaing ...",Positive,Happy
2,"barang bagus, berfungsi dengan baik, seler ram...",Positive,Happy
3,bagus sesuai harapan penjual nya juga ramah. t...,Positive,Happy
4,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",Positive,Happy


In [95]:
df.columns = ['review', 'sentiment', 'emotion']

In [96]:
df.isnull().sum()

,0
review,0
sentiment,0
emotion,0


In [97]:
# Data Cleaning TF-IDF dan FastText

def clean_text(text):
  text = str(text)
  text = text.lower()
  text = re.sub(r"http\S+", "", text)
  text = re.sub(r"www\S+", "", text)
  text = re.sub(r"@\w+", "", text)
  text = re.sub(r"#\w+", "", text)
  text = re.sub(r"[^a-zA-Z\s]", " ", text)
  text = re.sub(r"\s+", " ", text)
  return text.strip()

In [98]:
df['clean_review'] = df['review'].apply(clean_text)

In [99]:
df[['review', 'clean_review']].head()

,review,clean_review
0,Alhamdulillah berfungsi dengan baik. Packaging...,alhamdulillah berfungsi dengan baik packaging ...
1,"barang bagus dan respon cepat, harga bersaing ...",barang bagus dan respon cepat harga bersaing d...
2,"barang bagus, berfungsi dengan baik, seler ram...",barang bagus berfungsi dengan baik seler ramah...
3,bagus sesuai harapan penjual nya juga ramah. t...,bagus sesuai harapan penjual nya juga ramah tr...
4,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",barang bagus pengemasan aman dapat berfungsi d...


In [100]:
sentiment_encoder = LabelEncoder()
df['sentiment_label'] = sentiment_encoder.fit_transform(df['sentiment'])

In [101]:
dict(zip(sentiment_encoder.classes_, range(len(sentiment_encoder.classes_))))

{'Negative': 0, 'Positive': 1}

In [102]:
emotion_encoder = LabelEncoder()
df['emotion_label'] = emotion_encoder.fit_transform(df['emotion'])

In [103]:
dict(zip(emotion_encoder.classes_, range(len(emotion_encoder.classes_))))

{'Anger': 0, 'Fear': 1, 'Happy': 2, 'Love': 3, 'Sadness': 4}

In [104]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['sentiment_label'])

In [105]:
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['sentiment_label'])

In [106]:
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 3780
Val: 810
Test: 810


In [107]:
SAVE_PATH = "/content/ProjectNLP/dataset/preprocessed/"

os.makedirs(SAVE_PATH, exist_ok=True)
train_df.to_csv(f"{SAVE_PATH}/train.csv", index=False)
val_df.to_csv(f"{SAVE_PATH}/val.csv", index=False)
test_df.to_csv(f"{SAVE_PATH}/test.csv", index=False)

In [113]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   dataset/preprocessed/test.csv
	new file:   dataset/preprocessed/train.csv
	new file:   dataset/preprocessed/val.csv



In [114]:
!git add .

In [122]:
!git config --global user.name "discaaa"
!git config --global user.email "adisca.gandawidjaja@binus.ac.id"

In [123]:
!git add .

!git commit -m "updated preprocessing notebook"

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [124]:
!git remote -v

origin	https://github.com/asipnana/ProjectNLP.git (fetch)
origin	https://github.com/asipnana/ProjectNLP.git (push)


In [125]:
!git push -u origin main

fatal: could not read Username for 'https://github.com': No such device or address
